# AMML Ranking Competition 2026 — solution

System note: prefer **Kaggle Notebook GPU** (`kaggle_kernel/amml-ranking-2026-gpu.ipynb`). Local CPU works but CE fine-tune is slow.

Pipeline:
1. Lexical / structural features (BM25, TF-IDF, overlap, passage metadata)
2. GroupKFold OOF fine-tune of `DiTy/cross-encoder-russian-msmarco` (sentence-transformers CrossEncoder)
3. CatBoost YetiRank on features + CE scores

Fine-tuning uses **only** competition `train.csv` labels. No external labeled datasets.

In [ ]:
import platform, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from src.preprocessing import prepare_frame
from src.features import assemble_features
from src.train import cross_validate, train_full, predict, get_feature_cols
from src.ce_train import oof_st_cross_encoder_scores
from src.metrics import mean_ndcg_at_k

print(platform.platform(), sys.version)
print('cuda', torch.cuda.is_available())

DATA = Path('data')
MODELS = Path('models')
OUT = Path('results')
MODELS.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

train = prepare_frame(pd.read_csv(DATA / 'train.csv'))
test = prepare_frame(pd.read_csv(DATA / 'test.csv'))
print(train.shape, test.shape)

In [ ]:
feat_tr, feat_te = assemble_features(train, test)

CE_MODEL = 'DiTy/cross-encoder-russian-msmarco'
ce_epochs = 2 if torch.cuda.is_available() else 1
ce_bs = 32 if torch.cuda.is_available() else 8
ce_oof, ce_te = oof_st_cross_encoder_scores(
    train, test,
    models_dir=MODELS,
    model_name=CE_MODEL,
    epochs=ce_epochs,
    batch_size=ce_bs,
    max_len=256,
    n_splits=5,
)

feat_tr = feat_tr.copy(); feat_te = feat_te.copy()
feat_tr['ce_score'] = ce_oof
feat_te['ce_score'] = ce_te
for df in (feat_tr, feat_te):
    g = df.groupby('query_id')['ce_score']
    df['ce_score_qrank'] = g.rank(method='average', ascending=False)
    mean = g.transform('mean')
    std = g.transform('std').replace(0, np.nan)
    df['ce_score_qz'] = ((df['ce_score'] - mean) / std).fillna(0.0)

labels = train['relevance']
cv, oof = cross_validate(feat_tr, labels, n_splits=5)
model = train_full(feat_tr, labels, MODELS, best_iterations=1500)
feature_cols = get_feature_cols(feat_tr.assign(relevance=labels))
pred = predict(model, feat_te, feature_cols)

scored = train[['id', 'query_id', 'relevance']].copy()
scored['prediction'] = oof
print('OOF NDCG@10', mean_ndcg_at_k(scored), 'CV', cv)

sub = test[['id']].copy()
sub['prediction'] = pred
sub.to_csv(OUT / 'submission.csv', index=False)
print('wrote', OUT / 'submission.csv', len(sub))
sub.head()